In [1]:
using CMPSExcitations

In [1]:
# canonical basis
function projection_matrix1(D, R)
    Dr, M = eigen(R)

    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

function projection_matrix2(D, R)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    Dr, M = eigen(R)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        dD1 = view(e, 1:D)
        dD2 = view(e, D+1:2*D)
        X = zeros(D, D)

        k = 2D + 1
        for i in 1:D, j in 1:D
            if i != j
                X[i, j] = e[k] # sets the one hot vector
                k += 1
            end
        end

        Dr = Diagonal(Dr)
        W1 = M * ((X * Dr - Dr * X) + Diagonal(dD1)) / M
        W2 = M * ((X * Dr - Dr * X) + Diagonal(dD2)) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

function projection_matrix3(D, R)
    Dr, M = eigen(R)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E - M * Diagonal(F) / M
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

# canonical basis
function excitation_matrix(Heff, D)
    dim = 2 * D^2
    M = zeros(ComplexF64, dim, dim)

    for j in 1:dim
        e = zeros(ComplexF64, dim)
        e[j] = 1.0
        W1 = reshape(view(e, 1:D^2), D, D)
        W2 = reshape(view(e, D^2+1:dim), D, D)
        W1p, W2p = Heff((Constant(W1), Constant(W2)))
        M[:, j] = vcat(vec(W1p[]), vec(W2p[]))
    end

    return M
end

excitation_matrix (generic function with 1 method)

In [3]:
Hsingle_ll(c, μ) = ∫(∂ψ̂' * ∂ψ̂ - μ * ψ̂' * ψ̂ + c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hsingle(c, μ) = ∫(2 * ∂ψ̂' * ∂ψ̂ - 2 * μ * ψ̂' * ψ̂ + 4 * c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
# Hcoupled(c, μ) = ∫(
#     (∂ψ̂₁' * ∂ψ̂₁ - μ * ψ̂₁' * ψ̂₁ + c * (ψ̂₁')^2 * ψ̂₁^2 +
#      ∂ψ̂₂' * ∂ψ̂₂ - μ * ψ̂₂' * ψ̂₂ + c * (ψ̂₂')^2 * ψ̂₂^2 +
#      2 * c * (ψ̂₁') * (ψ̂₂') * ψ̂₂ * ψ̂₁), (-Inf, +Inf));
Hcoupled(c, μ) = ∫(
    (∂ψ̂₁' * ∂ψ̂₁ - μ * ψ̂₁' * ψ̂₁ + c * (ψ̂₁')^2 * ψ̂₁^2 +
     ∂ψ̂₂' * ∂ψ̂₂ - μ * ψ̂₂' * ψ̂₂ + c * (ψ̂₂')^2 * ψ̂₂^2 +
     c * (ψ̂₁') * (ψ̂₂') * ψ̂₂ * ψ̂₁ + c * (ψ̂₂') * (ψ̂₁') * ψ̂₁ * ψ̂₂), (-Inf, +Inf));

In [4]:
c, μ = 10., 5.
tol = 1e-10

Ds = [4]
D = maximum(Ds)

HLL = Hsingle(c, μ)
@time stateLL = find_groundstate(Ds, HLL, YangGaudinCMPS, optalg=LBFGS(80; verbosity=1, maxiter=7000, gradtol=tol), gradtol=tol)
println("Energy density: ", expval(HLL.h, stateLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateLL)[], "\n Order parameter: ", expval(ψ̂, stateLL)[])

# -----
stateCLL = InfiniteCMPS(stateLL.Q, (stateLL.Rs[1], stateLL.Rs[1]));
HCLL = Hcoupled(c, μ)
println("Energy density: ", expval(HCLL.h, stateCLL)[], "\nParticle density: ", expval(ψ̂₁' * ψ̂₁ + ψ̂₂' * ψ̂₂, stateCLL)[], "\nDensity imbalance: ", expval(ψ̂₁' * ψ̂₁ - ψ̂₂' * ψ̂₂, stateCLL)[])

Optimizing D=4


┌ Warning: The function `inner` is not implemented for (values of) type `Tuple{Constant{Matrix{Float64}}, Constant{Matrix{Float64}}}`;
│ this fallback will disappear in future versions of VectorInterface.jl
└ @ VectorInterface /home/ashankar/.julia/packages/VectorInterface/J6qCR/src/fallbacks.jl:196
┌ Warning: The function `scale` is not implemented for (values of) type `Tuple{Constant{Matrix{Float64}}, Float64}`;
│ this fallback will disappear in future versions of VectorInterface.jl
└ @ VectorInterface /home/ashankar/.julia/packages/VectorInterface/J6qCR/src/fallbacks.jl:67
┌ Warning: The function `add!!` is not implemented for (values of) type `Tuple{Constant{Matrix{Float64}}, Constant{Matrix{Float64}}, Float64, VectorInterface.One}`;
│ this fallback will disappear in future versions of VectorInterface.jl
└ @ VectorInterface /home/ashankar/.julia/packages/VectorInterface/J6qCR/src/fallbacks.jl:163
┌ Warning: The function `scalartype` is not implemented for (values of) type `Constant

D = 4 | YangGaudinCMPS{Constant{Matrix{Float64}}, 1}
 10.003921 seconds (57.78 M allocations: 2.822 GiB, 2.40% gc time, 99.09% compilation time: <1% of which was recompilation)
---------------
 12.814008 seconds (88.46 M allocations: 4.356 GiB, 3.05% gc time, 99.29% compilation time: <1% of which was recompilation)
Energy density: -2.7347478175232247

┌ Info: YangGaudinCMPS ground state: converged after 119 iterations: e = -2.734747817523, ‖∇e‖ = 8.9633e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:118



 Particle density: 0.43244359486794276
 Order parameter: 0.4386898377574924
Energy density: -2.734747817523213
Particle density: 0.8648871897358867
Density imbalance: 0.0


In [5]:
function leftcanonical(state, cmps=false)
    leftgauge!(state)
    r = rightenv(state)[1][]
    D, U = eigen(r)
    Q = U \ state.Q[] * U
    R = U \ state.Rs[1][] * U
    return (cmps) ? InfiniteCMPS(Constant(Q), (Constant(R), Constant(R))) : (Q, R)
end

leftcanonical (generic function with 2 methods)

In [6]:
# common setup
stateCLL = leftcanonical(stateCLL, true)
p = 0 # momentum
R = stateCLL.Rs[1][]
D = size(R, 1) # R = MDᵣ/M
space = InfiniteCMPSExcitationSpace(p, stateCLL, stateCLL)
H = excitation_matrix(excitation_operator(HCLL, space), D);

In [7]:
P = projection_matrix1(D, R)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))


[0.9372912379356885, 3.54324265454746, 7.65635713328682, 9.294323418024328, 11.024621605368528]


In [8]:
P = Matrix(qr(projection_matrix1(D, R)).Q)
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[0.9372912379357136, 3.5432426545474285, 7.656357133286829, 9.294323418024309, 11.02462160536857]


In [9]:
P = projection_matrix2(D, R)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))

[0.9372912379999896, 3.54324265459561, 7.656357133344012, 9.294323418114436, 11.0246216055838]


In [10]:
P = Matrix(qr(projection_matrix2(D, R)).Q)
@assert P' * P ≈ I
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[0.937291237935695, 3.543242654547432, 7.656357133286844, 9.29432341802433, 11.02462160536859]


In [11]:
P = projection_matrix3(D, R)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))

[0.9372912379356917, 3.5432426545474414, 7.656357133286821, 9.294323418024325, 11.02462160536856]


In [12]:
P = Matrix(qr(projection_matrix3(D, R)).Q)
@assert P' * P ≈ I
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[0.9372912379356776, 3.5432426545474454, 7.656357133286828, 9.294323418024316, 11.024621605368576]


QR instead of geneigsolve seems to be consistently equivalent as expected. Different parametrization seem to agree as well. However, the same parametrization gives different results based on the ground state which presumably only varies by gauge.. ??? Specifically the projector seems to be the issue.